# Notebook für die Daten Transformation und laden in data/silver

## Imports

In [186]:
import pandas as pd
import requests
from tqdm import tqdm
import os
from datetime import datetime, timedelta

## load data from data/bronze

In [187]:
lfa1 = pd.read_csv("/workspace/data/bronze/LFA1.csv", sep=";")
rechnungen_sap_2023 = pd.read_csv("/workspace/data/bronze/Rechnungen_SAP_2023.csv", sep=";")
rechnungen_sap_2024 = pd.read_csv("/workspace/data/bronze/Rechnungen_SAP_2024.csv", sep="|")

data_dict = {
    "LFA1": lfa1,
    "Rechnungen_SAP_2023": rechnungen_sap_2023,
    "Rechnungen_SAP_2024": rechnungen_sap_2024
}

## Ausgabe der ersten Zeilen

In [188]:
pd.set_option('display.max_columns', None)  # Show all columns in the output
pd.set_option('display.max_rows', None)  # Show all rows in the output

In [189]:
for i in data_dict.keys():
    print(f"Data from {i}:")
    print("Anzahl Spalten:", data_dict[i].shape[1], "Anzahl Zeilen:", data_dict[i].shape[0])
    display(data_dict[i].head())
    print("\n")

Data from LFA1:
Anzahl Spalten: 2 Anzahl Zeilen: 68252


,Lieferant,Name
0,L0194,Deutsche Post GmbH & Co.KG
1,L0027,ROBERT BOSCH
2,L0097,Engelbert und Strauss
3,L0028,Maschinenbau GmbH & Co. KG
4,L0028,Maschinenbau GmbH & Co. KG




Data from Rechnungen_SAP_2023:
Anzahl Spalten: 22 Anzahl Zeilen: 34170


,Rechnungsnummer,Bestellnummer,Belegdatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Spend,Währung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag
0,R-81402060,NaN,2023-01-01,Versand Kundenprämien,L0194,SK_000008,Porto und Brief,2,Vertriebsgesellschaft GmbH,NaN,NaN,14020000,1,NaN,NaN,683.40,EUR,14,14 Tage netto,2023-01-15,2023-01-09,NaN
1,R-60983007,NaN,2023-01-01,Zylinderschraube DIN,L0027,SK_000003,Maschinenteile und Ersatzte.,1,Produktion Bonn GmbH,NaN,NaN,23110000,1,NaN,NaN,211.53,EUR,30,30 Tage netto,2023-01-31,2023-02-10,V-74154614
2,R-51330234,NaN,2023-01-01,Dummy Klaus Ludwig Leopard-Sicherheitsschuhe T...,L0097,SK_000004,"Arbeitssicherheit, Ausr.",2,Vertriebsgesellschaft GmbH,NaN,NaN,40250703,1,NaN,NaN,8655.51,EUR,30,30 Tage; 1% Skonto,2023-01-31,2023-01-25,NaN
3,R-62280354,NaN,2023-01-01,SPIBO premium,L0028,SK_000003,Maschinenteile und Ersatzte.,4,Auslandsgesellschaft SRO,NaN,NaN,21182301,1,NaN,NaN,10937.30,EUR,14,"14 Tage; 1,5% Skonto",2023-01-15,2023-01-11,NaN
4,R-21995158,NaN,2023-01-01,SPIBO premium,L0028,SK_000003,Maschinenteile und Ersatzte.,4,Auslandsgesellschaft SRO,NaN,NaN,21182301,1,NaN,NaN,10991.05,EUR,14,"14 Tage; 1,5% Skonto",2023-01-15,2023-01-29,NaN




Data from Rechnungen_SAP_2024:
Anzahl Spalten: 22 Anzahl Zeilen: 34082


,Rechnungsnummer_SAP,Bestellnummer,BelegDatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Rechnungswert,Rechnungswährung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag
0,R-19934029,NaN,01/01/2024,Papier weiß A3,L0318,SK_000002,NaN,3,Verwaltung Demo AG,NaN,NaN,24260604,1,NaN,NaN,1217.95,USD,30,30 Tage; 1% Skonto,2024-01-31,2024-01-28,NaN
1,R-20408868,NaN,01/01/2024,Zylinderschraube,L0028,NaN,Maschinenteile und Ersatzte.,3,Verwaltung Demo AG,NaN,NaN,23110000,1,NaN,NaN,592.38,USD,14,"14 Tage; 1,5% Skonto",2024-01-15,2024-02-11,NaN
2,R-53946015,NaN,01/01/2024,Zylinderschraube DIN,L0028,NaN,Maschinenteile und Ersatzte.,4,Auslandsgesellschaft SRO,NaN,NaN,23110000,1,NaN,NaN,419.30,USD,14,"14 Tage; 1,5% Skonto",2024-01-15,2024-01-09,NaN
3,R-49696293,NaN,01/01/2024,"Test Runddrahtbürsten Ø x-mm Edelstdr. gew.,mm...",L0028,NaN,Maschinenteile und Ersatzte.,3,Verwaltung Demo AG,NaN,NaN,21190605,1,NaN,NaN,2804.05,USD,14,"14 Tage; 1,5% Skonto",2024-01-15,2024-01-09,NaN
4,R-45623784,NaN,01/01/2024,Gas Pauschale September,L0315,SK_000020,NaN,4,Auslandsgesellschaft SRO,NaN,NaN,26040201,1,NaN,NaN,34259.35,USD,30,30 Tage netto,2024-01-31,2024-01-26,V-27993028


In [190]:
# Ausgabe der dtypes
for i in data_dict.keys():
    print(f"Data from {i}:")
    display(data_dict[i].dtypes)
    print("\n")

Data from LFA1:


Lieferant    object
Name         object
dtype: object



Data from Rechnungen_SAP_2023:


Rechnungsnummer          object
Bestellnummer            object
Belegdatum               object
Beschreibung             object
Lieferant_Nummer         object
Sachkonto-Nummer         object
Sachkonto-Name           object
Buchungskreis-Nummer      int64
Buchungskreis-Name       object
Material-Nummer          object
Material-Name            object
ECLASS-identifier         int64
Menge                     int64
Mengeneinheit            object
Stückpreis              float64
Spend                   float64
Währung                  object
Zahlungsziel-Tage         int64
Zahlungsbedingung        object
Zahlungsfrist            object
Zahlungsdatum            object
Rahmenvertrag            object
dtype: object



Data from Rechnungen_SAP_2024:


Rechnungsnummer_SAP      object
Bestellnummer            object
BelegDatum               object
Beschreibung             object
Lieferant_Nummer         object
Sachkonto-Nummer         object
Sachkonto-Name           object
Buchungskreis-Nummer      int64
Buchungskreis-Name       object
Material-Nummer          object
Material-Name            object
ECLASS-identifier         int64
Menge                     int64
Mengeneinheit            object
Stückpreis              float64
Rechnungswert           float64
Rechnungswährung         object
Zahlungsziel-Tage         int64
Zahlungsbedingung        object
Zahlungsfrist            object
Zahlungsdatum            object
Rahmenvertrag            object
dtype: object

# Genauere Blick auf LFA1

In [191]:
# wie viele unique Lieferanten gibt es
print(f"Anzahl Unique Lieferanten: {lfa1['Lieferant'].nunique()}")

lieferanten = lfa1.groupby("Lieferant")
# Ausgabe von groupby("Lieferant")
for i, (name, group) in enumerate(lieferanten):
    print(f"\nGruppe: {name}")
    display(group.head()) 
    if i == 4:
        break

Anzahl Unique Lieferanten: 219

Gruppe: L0003


,Lieferant,Name
380,L0003,TATA STEEL EUROPE
940,L0003,TATA STEEL EUROPE
1243,L0003,TATA STEEL EUROPE
2516,L0003,TATA STEEL EUROPE
2517,L0003,TATA STEEL EUROPE



Gruppe: L0005


,Lieferant,Name
72,L0005,Krupp Essen Deutschland
76,L0005,Krupp Essen Deutschland
154,L0005,Krupp Essen Deutschland
225,L0005,Krupp Essen Deutschland
230,L0005,Krupp Essen Deutschland



Gruppe: L0006


,Lieferant,Name
413,L0006,KRUPP ESSEN GERMANY
640,L0006,KRUPP ESSEN GERMANY
700,L0006,KRUPP ESSEN GERMANY
1829,L0006,KRUPP ESSEN GERMANY
2967,L0006,KRUPP ESSEN GERMANY



Gruppe: L0007


,Lieferant,Name
166,L0007,Krupp Essen GmbH
214,L0007,Krupp Essen GmbH
226,L0007,Krupp Essen GmbH
366,L0007,Krupp Essen GmbH
401,L0007,Krupp Essen GmbH



Gruppe: L0008


,Lieferant,Name
66,L0008,Krupp+Essen
67,L0008,Krupp+Essen
82,L0008,Krupp+Essen
152,L0008,Krupp+Essen
367,L0008,Krupp+Essen


### Prüfen ob alle Name in Lieferant gleich sind

In [192]:
# prüfen ob alle Name in Lieferant gleich sind

# Gruppen prüfen, bei denen nicht nur ein eindeutiger Name vorkommt
inkonsistente = []

for lieferant_id, group in lieferanten:
    unique_names = group["Name"].unique()
    print(f"Lieferant {lieferant_id} hat {len(unique_names)} eindeutige Namen.")
    if len(unique_names) > 1:
        inkonsistente.append((lieferant_id, unique_names))

# Ergebnis anzeigen
if inkonsistente:
    for lieferant_id, names in inkonsistente:
        print(f"Lieferant {lieferant_id} hat mehrere Namen: {names}")
else:
    print("Es gibt keine inkonsistenten Lieferanten.")

Lieferant L0003 hat 1 eindeutige Namen.
Lieferant L0005 hat 1 eindeutige Namen.
Lieferant L0006 hat 1 eindeutige Namen.
Lieferant L0007 hat 1 eindeutige Namen.
Lieferant L0008 hat 1 eindeutige Namen.
Lieferant L0009 hat 1 eindeutige Namen.
Lieferant L0012 hat 1 eindeutige Namen.
Lieferant L0023 hat 1 eindeutige Namen.
Lieferant L0026 hat 1 eindeutige Namen.
Lieferant L0027 hat 1 eindeutige Namen.
Lieferant L0028 hat 1 eindeutige Namen.
Lieferant L0029 hat 1 eindeutige Namen.
Lieferant L0030 hat 1 eindeutige Namen.
Lieferant L0031 hat 1 eindeutige Namen.
Lieferant L0032 hat 1 eindeutige Namen.
Lieferant L0033 hat 1 eindeutige Namen.
Lieferant L0034 hat 1 eindeutige Namen.
Lieferant L0035 hat 1 eindeutige Namen.
Lieferant L0036 hat 1 eindeutige Namen.
Lieferant L0037 hat 1 eindeutige Namen.
Lieferant L0038 hat 1 eindeutige Namen.
Lieferant L0039 hat 1 eindeutige Namen.
Lieferant L0040 hat 1 eindeutige Namen.
Lieferant L0041 hat 1 eindeutige Namen.
Lieferant L0042 hat 1 eindeutige Namen.


### Drop duplizierte Lieferanten

In [193]:
def drop_duplicates_lieferanten(df: pd.DataFrame) -> pd.DataFrame:
    """
    Entfernt duplizierte Lieferanten basierend auf den Spalten 'Lieferant' und 'Name'.

    Args:
        df (pd.DataFrame): Der DataFrame, aus dem Duplikate entfernt werden sollen.

    Returns:
        pd.DataFrame: Ein DataFrame ohne duplizierte Lieferanten.
    """
    return df.drop_duplicates(subset=["Lieferant", "Name"], keep='first').reset_index(drop=True)

In [194]:
lfa1_cleaned = drop_duplicates_lieferanten(lfa1)

In [195]:
display(lfa1_cleaned)

,Lieferant,Name
0,L0194,Deutsche Post GmbH & Co.KG
1,L0027,ROBERT BOSCH
2,L0097,Engelbert und Strauss
3,L0028,Maschinenbau GmbH & Co. KG
4,L0193,Deutsche Post GmbH
5,L0026,Robert Bosch GmbH
6,L0190,Deutsche Post AG
7,L0192,Deutsche Post Germany
8,L0031,Heinrichs AG
9,L0063,Lämmermeier Edelstahl GmbH


# Genauere Blick auf SAP Rechnungen 2023

In [196]:
# gibt es fehlende Werte in den Lieferantendaten
display(rechnungen_sap_2023.isna().sum())


Rechnungsnummer             0
Bestellnummer           28208
Belegdatum                  0
Beschreibung                0
Lieferant_Nummer            0
Sachkonto-Nummer            0
Sachkonto-Name              0
Buchungskreis-Nummer        0
Buchungskreis-Name          0
Material-Nummer         28208
Material-Name           28208
ECLASS-identifier           0
Menge                       0
Mengeneinheit           28208
Stückpreis              28208
Spend                       0
Währung                     0
Zahlungsziel-Tage           0
Zahlungsbedingung           0
Zahlungsfrist               0
Zahlungsdatum               0
Rahmenvertrag           27275
dtype: int64

### Merge lieferantenname

In [197]:

def merge_lieferanten_names(df: pd.DataFrame, lfa1_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge Lieferanten-Namen aus dem LFA1 DataFrame in das Rechnungen DataFrame.

    Args:
        df (pd.DataFrame): Das Rechnungen DataFrame, in das die Namen eingefügt werden sollen.
        lfa1_df (pd.DataFrame): Das LFA1 DataFrame, das die Lieferantennamen enthält.

    Returns:
        pd.DataFrame: Ein DataFrame mit den Lieferantennamen aus dem LFA1 DataFrame.
    """
    return df.merge(lfa1_df[['Lieferant', 'Name']], left_on='Lieferant_Nummer', right_on='Lieferant', how='left').drop(columns='Lieferant')

In [198]:
rechnungen_sap_2023_lieferantenname = merge_lieferanten_names(rechnungen_sap_2023, lfa1_cleaned).rename(columns={'Name': 'Lieferantenname'})


In [199]:
for i, (name, group) in enumerate(rechnungen_sap_2023_lieferantenname.groupby("Lieferant_Nummer")):
    print(f"\nGruppe: {name}")
    display(group.head(1))
    if i == 4:
        break


Gruppe: L0003


,Rechnungsnummer,Bestellnummer,Belegdatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Spend,Währung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag,Lieferantenname
380,R-72788362,B-88391834,2023-01-04,Stahl legiert wellbl. IDL737 500,L0003,SK_000001,Wareneinkauf,2,Vertriebsgesellschaft GmbH,M-016,Stahl legiert wellbl. IDL737 500,35060105,174,Rolle,850.0,147900.0,EUR,180,180 Tage netto,2023-07-03,2023-08-01,NaN,TATA STEEL EUROPE



Gruppe: L0005


,Rechnungsnummer,Bestellnummer,Belegdatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Spend,Währung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag,Lieferantenname
72,R-10981168,B-44879492,2023-01-01,Draht Max 0-15,L0005,SK_000001,Wareneinkauf,5,US plant Inc.,M-017,Draht Max 0-15,35060601,156,Meter,80.0,12480.0,EUR,60,60 Tage netto,2023-03-02,2023-03-01,NaN,Krupp Essen Deutschland



Gruppe: L0006


,Rechnungsnummer,Bestellnummer,Belegdatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Spend,Währung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag,Lieferantenname
413,R-00347148,B-51842853,2023-01-05,Stahl legiert wellbl. IDL737 500,L0006,SK_000001,Wareneinkauf,3,Verwaltung Demo AG,M-016,Stahl legiert wellbl. IDL737 500,35060105,261,Rolle,850.0,221850.0,EUR,14,14 Tage netto,2023-01-19,2023-02-17,V-34844011,KRUPP ESSEN GERMANY



Gruppe: L0007


,Rechnungsnummer,Bestellnummer,Belegdatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Spend,Währung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag,Lieferantenname
166,R-85753889,B-26534037,2023-01-02,Draht CT8 SF 0-10,L0007,SK_000001,Wareneinkauf,5,US plant Inc.,M-021,Draht CT8 SF 0-10,35060601,258,Meter,65.0,16770.0,EUR,180,180 Tage netto,2023-07-01,2023-06-28,V-11169730,Krupp Essen GmbH



Gruppe: L0008


,Rechnungsnummer,Bestellnummer,Belegdatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Spend,Währung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag,Lieferantenname
66,R-35114858,B-26998748,2023-01-01,Draht CT8 SF 0-10,L0008,SK_000001,Wareneinkauf,4,Auslandsgesellschaft SRO,M-021,Draht CT8 SF 0-10,35060601,103,Meter,65.0,6695.0,USD,0,Sofort ohne Abzug,2023-01-01,2021-12-27,V-81587600,Krupp+Essen


### Überprüfen wie viele unterschiedliche Währungen es gibt

In [200]:

display(rechnungen_sap_2023['Währung'].unique())
display(rechnungen_sap_2024['Rechnungswährung'].unique())

for währung in rechnungen_sap_2023['Währung'].unique():
    print(f"Währung: {währung}")
    display(rechnungen_sap_2023[rechnungen_sap_2023['Währung'] == währung].head())


array(['EUR', 'USD', 'GBP'], dtype=object)

array(['USD', 'EUR', 'GBP'], dtype=object)

Währung: EUR


,Rechnungsnummer,Bestellnummer,Belegdatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Spend,Währung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag
0,R-81402060,NaN,2023-01-01,Versand Kundenprämien,L0194,SK_000008,Porto und Brief,2,Vertriebsgesellschaft GmbH,NaN,NaN,14020000,1,NaN,NaN,683.40,EUR,14,14 Tage netto,2023-01-15,2023-01-09,NaN
1,R-60983007,NaN,2023-01-01,Zylinderschraube DIN,L0027,SK_000003,Maschinenteile und Ersatzte.,1,Produktion Bonn GmbH,NaN,NaN,23110000,1,NaN,NaN,211.53,EUR,30,30 Tage netto,2023-01-31,2023-02-10,V-74154614
2,R-51330234,NaN,2023-01-01,Dummy Klaus Ludwig Leopard-Sicherheitsschuhe T...,L0097,SK_000004,"Arbeitssicherheit, Ausr.",2,Vertriebsgesellschaft GmbH,NaN,NaN,40250703,1,NaN,NaN,8655.51,EUR,30,30 Tage; 1% Skonto,2023-01-31,2023-01-25,NaN
3,R-62280354,NaN,2023-01-01,SPIBO premium,L0028,SK_000003,Maschinenteile und Ersatzte.,4,Auslandsgesellschaft SRO,NaN,NaN,21182301,1,NaN,NaN,10937.30,EUR,14,"14 Tage; 1,5% Skonto",2023-01-15,2023-01-11,NaN
4,R-21995158,NaN,2023-01-01,SPIBO premium,L0028,SK_000003,Maschinenteile und Ersatzte.,4,Auslandsgesellschaft SRO,NaN,NaN,21182301,1,NaN,NaN,10991.05,EUR,14,"14 Tage; 1,5% Skonto",2023-01-15,2023-01-29,NaN


Währung: USD


,Rechnungsnummer,Bestellnummer,Belegdatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Spend,Währung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag
15,R-43006533,NaN,2023-01-01,Dichtung 18.97162,L0027,SK_000003,Maschinenteile und Ersatzte.,4,Auslandsgesellschaft SRO,NaN,NaN,23070000,1,NaN,NaN,447.15,USD,30,30 Tage netto,2023-01-31,2023-01-28,V-81636418
16,R-05660076,NaN,2023-01-01,Zylinderschraube DIN,L0028,SK_000003,Maschinenteile und Ersatzte.,3,Verwaltung Demo AG,NaN,NaN,23110000,1,NaN,NaN,496.65,USD,14,"14 Tage; 1,5% Skonto",2023-01-15,2023-01-11,NaN
17,R-19838123,NaN,2023-01-01,Reinigung Büro in Bonn,L0169,SK_000022,Reinigungsdienstleistung,2,Vertriebsgesellschaft GmbH,NaN,NaN,25290100,1,NaN,NaN,62207.79,USD,30,30 Tage; 1% Skonto,2023-01-31,2023-01-29,NaN
18,R-28385022,NaN,2023-01-01,Reinigung Bürogebäude in Bonn - Q1,L0161,SK_000022,Reinigungsdienstleistung,4,Auslandsgesellschaft SRO,NaN,NaN,25290100,1,NaN,NaN,84586.12,USD,0,Sofort ohne Abzug,2023-01-01,2021-12-31,NaN
19,R-63273934,NaN,2023-01-01,Schlitzschraubenzieher,L0026,SK_000003,Maschinenteile und Ersatzte.,5,US plant Inc.,NaN,NaN,21040490,1,NaN,NaN,551.20,USD,14,14 Tage netto,2023-01-15,2023-01-14,NaN


Währung: GBP


,Rechnungsnummer,Bestellnummer,Belegdatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Spend,Währung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag
15742,R-68266911,NaN,2023-06-18,Bu1 ZINKEIMER LITER,L0236,SK_000009,Logistik,1,Produktion Bonn GmbH,NaN,NaN,20050101,1,NaN,NaN,1453.75,GBP,14,14 Tage netto,2023-07-02,2023-07-04,NaN
16055,R-59091440,NaN,2023-06-21,Papier A3 120g /m² weiß Inkjet 100Bl/Pg,L0318,SK_000030,Bürobedarf,1,Produktion Bonn GmbH,NaN,NaN,24260604,1,NaN,NaN,789.27,GBP,30,30 Tage; 1% Skonto,2023-07-21,2023-07-15,NaN
16056,R-34217140,B-77649583,2023-06-21,DIN 933 A4-70 M 16x50 ISO 4017 Sechskantschrauben,L0026,SK_000003,Maschinenteile und Ersatzte.,5,US plant Inc.,M-004,DIN 933 A4-70 M 16x50 ISO 4017 Sechskantschrauben,23110115,125,Stück,3.0,375.00,GBP,14,14 Tage netto,2023-07-05,2023-07-18,NaN
16057,R-35388948,NaN,2023-06-21,Schraubendreher mit Bits,L0028,SK_000003,Maschinenteile und Ersatzte.,3,Verwaltung Demo AG,NaN,NaN,21040490,1,NaN,NaN,401.89,GBP,14,"14 Tage; 1,5% Skonto",2023-07-05,2023-07-02,NaN
16058,R-12664095,NaN,2023-06-21,Schraubendreher mit Bits,L0028,SK_000003,Maschinenteile und Ersatzte.,2,Vertriebsgesellschaft GmbH,NaN,NaN,21040490,1,NaN,NaN,887.55,GBP,14,"14 Tage; 1,5% Skonto",2023-07-05,2023-07-03,NaN


### Bauen eines Wechselkurs df

In [201]:

def get_exchange_rates_df(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Holt Wechselkurse von USD und GBP zu EUR für jeden Tag im angegebenen Zeitraum.
    
    Parameter:
    - start_date (str): Startdatum im Format "YYYY-MM-DD"
    - end_date (str): Enddatum im Format "YYYY-MM-DD"
    
    Rückgabe:
    - DataFrame mit Spalten: Datum, Wechselkurs_USD, Wechselkurs_GBP
    """
    start_date  = datetime.strptime(start_date, "%Y-%m-%d")
    end_date  = datetime.strptime(end_date, "%Y-%m-%d")
    
    data = []

    # Tagesweise iterieren
    current_date = start_date
    while current_date <= end_date:
        formatted_date = current_date.strftime("%Y-%m-%d")
        
        usd_rate, gbp_rate = None, None

        # USD → EUR
        resp_usd = requests.get(f"https://api.frankfurter.app/{formatted_date}?from=USD&to=EUR")
        if resp_usd.ok:
            usd_rate = resp_usd.json()["rates"]["EUR"]

        # GBP → EUR
        resp_gbp = requests.get(f"https://api.frankfurter.app/{formatted_date}?from=GBP&to=EUR")
        if resp_gbp.ok:
            gbp_rate = resp_gbp.json()["rates"]["EUR"]
        
        data.append({
            "Datum": formatted_date,
            "Wechselkurs_USD": usd_rate,
            "Wechselkurs_GBP": gbp_rate
        })
        
        current_date += timedelta(days=1)

        df = pd.DataFrame(data)

        if df.isnull().values.any():
            print(f"Fehlende Wechselkurse für das Datum: {formatted_date}")

    return df


In [ ]:
wechselkurse_df = get_exchange_rates_df("2023-01-01", "2024-12-31")
wechselkurse_df.head()

In [ ]:
wechselkurse_df.to_csv("/workspace/data/wechselkurse.csv", index=False, sep=";")

In [ ]:
wechselkurse_df = pd.read_csv("/workspace/data/silver/wechselkurse.csv", sep=";")

In [ ]:
wechselkurse_df.isna().sum()

Datum              0
Wechselkurs_USD    0
Wechselkurs_GBP    0
dtype: int64

### Inplausible Daten gefunden, es gibt Daten indem das Belegdatum später ist als das Zahlungsdatum und das Rechnungen von 2023 im Jahr 2021 bezahlt wurden. Es sieht als wären die Zahlungsdatum von 2021 um ein Jahr nach hinten geschoben sind also eigentlich vom Jahr 2022. Das wäre zu klären mit ein Domänexperte. Für den weiteren Verlauf werden die Rechnungen vom Jahr 2021 gelöscht.

In [ ]:
fehlerhafte_zeilen = rechnungen_sap_2023_lieferantenname[
    rechnungen_sap_2023_lieferantenname["Zahlungsdatum"] < rechnungen_sap_2023_lieferantenname["Belegdatum"]
]

display(fehlerhafte_zeilen[["Zahlungsdatum", "Belegdatum", "Lieferant_Nummer", "Lieferantenname", "Spend"]].sort_values(by="Zahlungsdatum").head(10))
print(f"Anzahl 'fehlerhafter' Zeilen: {len(fehlerhafte_zeilen)}")


,Zahlungsdatum,Belegdatum,Lieferant_Nummer,Lieferantenname,Spend
67,2021-12-26,2023-01-01,L0008,Krupp+Essen,11635.00
66,2021-12-27,2023-01-01,L0008,Krupp+Essen,6695.00
5,2021-12-28,2023-01-01,L0193,Deutsche Post GmbH,1154.69
152,2021-12-28,2023-01-02,L0008,Krupp+Essen,10335.00
283,2021-12-28,2023-01-03,L0193,Deutsche Post GmbH,1943.13
20,2021-12-29,2023-01-01,L0189,Deutsche Post,1388.20
107,2021-12-29,2023-01-02,L0193,Deutsche Post GmbH,1153.91
248,2021-12-29,2023-01-03,L0193,Deutsche Post GmbH,733.82
170,2021-12-30,2023-01-02,L0071,IFM Elektronic,22718.81
428,2021-12-30,2023-01-05,L0193,Deutsche Post GmbH,1795.05


Anzahl 'fehlerhafter' Zeilen: 2124


In [ ]:
# drop Zeilen mit Zahlungsdatum < 2022
def drop_invalid_payments(df: pd.DataFrame, min_date: str) -> pd.DataFrame:
    """
    Entfernt Zeilen mit einem Zahlungsdatum vor dem angegebenen Mindestdatum.

    Args:
        df (pd.DataFrame): Das DataFrame, aus dem die ungültigen Zahlungen entfernt werden sollen.
        min_date (str): Das Mindestdatum im Format 'YYYY-MM-DD'.

    Returns:
        pd.DataFrame: Ein DataFrame ohne ungültige Zahlungen.
    """

    return df[df["Zahlungsdatum"] >= min_date]

In [ ]:
rechnungen_sap_2023_lieferantenname_valid = drop_invalid_payments(rechnungen_sap_2023_lieferantenname, "2022-01-01")


In [ ]:
display(rechnungen_sap_2023_lieferantenname_valid[["Zahlungsdatum", "Belegdatum", "Lieferant_Nummer", "Lieferantenname", "Spend", "Währung"]].sort_values(by="Zahlungsdatum").head())


,Zahlungsdatum,Belegdatum,Lieferant_Nummer,Lieferantenname,Spend,Währung
423,2023-01-01,2023-01-05,L0060,Worrings Prüftechnik GmbH,605.85,EUR
426,2023-01-01,2023-01-05,L0161,Reinigung Fassbender,7855.90,EUR
430,2023-01-01,2023-01-05,L0193,Deutsche Post GmbH,862.46,EUR
449,2023-01-01,2023-01-05,L0189,Deutsche Post,1178.29,EUR
499,2023-01-01,2023-01-06,L0189,Deutsche Post,898.55,EUR


### Wechselkurs den Rechnungen Df hinzufügen

In [ ]:
def add_wechselkurse_to_rechnungen(rechnungen_df: pd.DataFrame, wechselkurse_df: pd.DataFrame, währung_column: str, zahlungsdatum_column: str) -> pd.DataFrame:
    """
    Fügt Wechselkurse zu den Rechnungsdaten hinzu.
    
    Args:
    - rechnungen_df (pd.DataFrame): DataFrame mit Rechnungsdaten
    - wechselkurse_df (pd.DataFrame): DataFrame mit Wechselkursen
    - währung_column (str): Der Name der Währungsspalte im Rechnungs-DataFrame
    - zahlungsdatum_column (str): Der Name der Zahlungsdatumsspalte im Rechnungs-DataFrame

    Returns:
    - pd.DataFrame: DataFrame mit hinzugefügten Wechselkursen
    """
    wechselkurs_liste = []

    for _, row in rechnungen_df.iterrows():
        datum = row[zahlungsdatum_column]
        währung = row[währung_column]

        if pd.isna(datum) or pd.isna(währung):
            print("Daten fehlen")
            break

        if währung == "EUR":
            wechselkurs_liste.append(1.0)
        else:
            kurs_row = wechselkurse_df[wechselkurse_df["Datum"] == datum]
            if kurs_row.empty:
                wechselkurs_liste.append(None)
            else:
                if währung == "USD":
                    wechselkurs_liste.append(kurs_row["Wechselkurs_USD"].values[0])
                elif währung == "GBP":
                    wechselkurs_liste.append(kurs_row["Wechselkurs_GBP"].values[0])
                else:
                    wechselkurs_liste.append(None)

    rechnungen_df["Wechselkurs"] = wechselkurs_liste

    return rechnungen_df

In [ ]:
rechnungen_sap_2023_lieferantenname_valid_wechselkurs = add_wechselkurse_to_rechnungen(rechnungen_sap_2023_lieferantenname_valid, wechselkurse_df, "Währung", "Zahlungsdatum")

/tmp/ipykernel_1149/2556523556.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rechnungen_df["Wechselkurs"] = wechselkurs_liste


In [ ]:
for i, (name, group) in enumerate(rechnungen_sap_2023_lieferantenname_valid_wechselkurs.groupby("Währung")):
    print(f"\nGruppe: {name}")
    display(group[["Zahlungsdatum", "Belegdatum", "Lieferant_Nummer", "Lieferantenname", "Spend", "Währung", "Wechselkurs"]].head(1))
    if i == 4:
        break


Gruppe: EUR


,Zahlungsdatum,Belegdatum,Lieferant_Nummer,Lieferantenname,Spend,Währung,Wechselkurs
0,2023-01-09,2023-01-01,L0194,Deutsche Post GmbH & Co.KG,683.4,EUR,1.0



Gruppe: GBP


,Zahlungsdatum,Belegdatum,Lieferant_Nummer,Lieferantenname,Spend,Währung,Wechselkurs
15742,2023-07-04,2023-06-18,L0236,Vereinigte Papierwarenfabriken GmbH,1453.75,GBP,1.1672



Gruppe: USD


,Zahlungsdatum,Belegdatum,Lieferant_Nummer,Lieferantenname,Spend,Währung,Wechselkurs
15,2023-01-28,2023-01-01,L0027,ROBERT BOSCH,447.15,USD,0.92039


In [ ]:
display(wechselkurse_df[wechselkurse_df["Datum"] == "2023-07-04"])
display(wechselkurse_df[wechselkurse_df["Datum"] == "2023-01-28"])

,Datum,Wechselkurs_USD,Wechselkurs_GBP
184,2023-07-04,0.91785,1.1672


,Datum,Wechselkurs_USD,Wechselkurs_GBP
27,2023-01-28,0.92039,1.1379


### Alle Rechnungen in Euro rechnen

In [ ]:
def calc_spendeuro(rechnungen_df: pd.DataFrame, spend_column: str) -> pd.DataFrame:
    """
    Berechnet den Betrag in Euro für jede Zeile basierend auf der Währung und dem Wechselkurs.

    Args:
        rechnungen_df (pd.DataFrame): Das DataFrame, das die Spalten 'Spend', 'Wechselkurs' enthält.
        spend_column (str): Der Name der Spalte, die den Betrag in der ursprünglichen Währung enthält.

    Returns:
        pd.DataFrame: Ein DataFrame mit einer neuen Spalte 'Spend_EUR'
    """
    rechnungen_df["Spend_EUR"] = rechnungen_df[spend_column] * rechnungen_df["Wechselkurs"]

    return rechnungen_df

In [ ]:
rechnungen_sap_2023_lieferantenname_valid_wechselkurs_spendeuro = calc_spendeuro(rechnungen_sap_2023_lieferantenname_valid_wechselkurs, "Spend")


/tmp/ipykernel_1149/1806863438.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rechnungen_df["Spend_EUR"] = rechnungen_df[spend_column] * rechnungen_df["Wechselkurs"]


In [ ]:
for i, (name, group) in enumerate(rechnungen_sap_2023_lieferantenname_valid_wechselkurs_spendeuro.groupby("Währung")):
    print(f"\nGruppe: {name}")
    display(group[["Zahlungsdatum", "Belegdatum", "Lieferant_Nummer", "Lieferantenname", "Spend", "Währung", "Wechselkurs", "Spend_EUR"]].head(1))
    if i == 4:
        break


Gruppe: EUR


,Zahlungsdatum,Belegdatum,Lieferant_Nummer,Lieferantenname,Spend,Währung,Wechselkurs,Spend_EUR
0,2023-01-09,2023-01-01,L0194,Deutsche Post GmbH & Co.KG,683.4,EUR,1.0,683.4



Gruppe: GBP


,Zahlungsdatum,Belegdatum,Lieferant_Nummer,Lieferantenname,Spend,Währung,Wechselkurs,Spend_EUR
15742,2023-07-04,2023-06-18,L0236,Vereinigte Papierwarenfabriken GmbH,1453.75,GBP,1.1672,1696.817



Gruppe: USD


,Zahlungsdatum,Belegdatum,Lieferant_Nummer,Lieferantenname,Spend,Währung,Wechselkurs,Spend_EUR
15,2023-01-28,2023-01-01,L0027,ROBERT BOSCH,447.15,USD,0.92039,411.552389


### Überprüfen der Sachkonto- nummer und name

In [ ]:
print(rechnungen_sap_2023_lieferantenname_valid_wechselkurs_spendeuro["Sachkonto-Nummer"].unique())
print(rechnungen_sap_2023_lieferantenname_valid_wechselkurs_spendeuro[["Sachkonto-Nummer", "Sachkonto-Name"]].drop_duplicates())


['SK_000008' 'SK_000003' 'SK_000004' 'SK_000009' 'SK_000022' 'SK_000007'
 'SK_000005' 'SK_000020' 'SK_000002' 'SK_000001' 'SK_000030' 'SK_000014'
 'SK_000015' 'SK_000029' 'SK_000019' 'SK_000011' 'SK_000016' 'SK_000021'
 'SK_000013']
     Sachkonto-Nummer                         Sachkonto-Name
0           SK_000008                        Porto und Brief
1           SK_000003           Maschinenteile und Ersatzte.
2           SK_000004               Arbeitssicherheit, Ausr.
7           SK_000009                               Logistik
17          SK_000022               Reinigungsdienstleistung
21          SK_000007                         Logistikbedarf
25          SK_000005               Aufwendungen für Gebäude
55          SK_000020                                Energie
69          SK_000002  Verrechnung vorherige Buchungsperiode
72          SK_000001                           Wareneinkauf
78          SK_000030                             Bürobedarf
131         SK_000014              

In [ ]:
print(rechnungen_sap_2023_lieferantenname_valid_wechselkurs_spendeuro["Sachkonto-Name"].unique())

['Porto und Brief' 'Maschinenteile und Ersatzte.'
 'Arbeitssicherheit, Ausr.' 'Logistik' 'Reinigungsdienstleistung'
 'Logistikbedarf' 'Aufwendungen für Gebäude' 'Energie'
 'Verrechnung vorherige Buchungsperiode' 'Wareneinkauf' 'Bürobedarf'
 'KFZ-Kosten' 'Dienstleistungen' 'Lizenzen' 'Beratungsleistungen'
 'Arbeitsmittel Hardware' 'Mediendienstleistung'
 'Aufwendungen für Entsorgung' 'Außendienst Hardware']


### Überprüfen des Belegdatum 

In [ ]:
print(rechnungen_sap_2023["Belegdatum"].unique())

['2023-01-01' '2023-01-02' '2023-01-03' '2023-01-04' '2023-01-05'
 '2023-01-06' '2023-01-07' '2023-01-08' '2023-01-09' '2023-01-10'
 '2023-01-11' '2023-01-12' '2023-01-13' '2023-01-14' '2023-01-15'
 '2023-01-16' '2023-01-17' '2023-01-18' '2023-01-19' '2023-01-20'
 '2023-01-21' '2023-01-22' '2023-01-23' '2023-01-24' '2023-01-25'
 '2023-01-26' '2023-01-27' '2023-01-28' '2023-01-29' '2023-01-30'
 '2023-01-31' '2023-02-01' '2023-02-02' '2023-02-03' '2023-02-04'
 '2023-02-05' '2023-02-06' '2023-02-07' '2023-02-08' '2023-02-09'
 '2023-02-10' '2023-02-11' '2023-02-12' '2023-02-13' '2023-02-14'
 '2023-02-15' '2023-02-16' '2023-02-17' '2023-02-18' '2023-02-19'
 '2023-02-20' '2023-02-21' '2023-02-22' '2023-02-23' '2023-02-24'
 '2023-02-25' '2023-02-26' '2023-02-27' '2023-02-28' '2023-03-01'
 '2023-03-02' '2023-03-03' '2023-03-04' '2023-03-05' '2023-03-06'
 '2023-03-07' '2023-03-08' '2023-03-09' '2023-03-10' '2023-03-11'
 '2023-03-12' '2023-03-13' '2023-03-14' '2023-03-15' '2023-03-16'
 '2023-03-

# Genauere Blick auf SAP Rechnungen 2024

In [ ]:
# gibt es fehlende Werte in den Lieferantendaten
display(rechnungen_sap_2024.isna().sum())

Rechnungsnummer_SAP         0
Bestellnummer           28262
BelegDatum                  0
Beschreibung                0
Lieferant_Nummer            0
Sachkonto-Nummer        16294
Sachkonto-Name          17788
Buchungskreis-Nummer        0
Buchungskreis-Name          0
Material-Nummer         28262
Material-Name           28262
ECLASS-identifier           0
Menge                       0
Mengeneinheit           28262
Stückpreis              28262
Rechnungswert               0
Rechnungswährung            0
Zahlungsziel-Tage           0
Zahlungsbedingung           0
Zahlungsfrist               0
Zahlungsdatum               0
Rahmenvertrag           27259
dtype: int64

### Anpassen des Belegdatum von mm/tt/jjjj zu jjjj-mm-tt

In [ ]:
print(rechnungen_sap_2024["BelegDatum"].unique())

['01/01/2024' '01/02/2024' '01/03/2024' '01/04/2024' '01/05/2024'
 '01/06/2024' '01/07/2024' '01/08/2024' '01/09/2024' '01/10/2024'
 '01/11/2024' '01/12/2024' '01/13/2024' '01/14/2024' '01/15/2024'
 '01/16/2024' '01/17/2024' '01/18/2024' '01/19/2024' '01/20/2024'
 '01/21/2024' '01/22/2024' '01/23/2024' '01/24/2024' '01/25/2024'
 '01/26/2024' '01/27/2024' '01/28/2024' '01/29/2024' '01/30/2024'
 '01/31/2024' '02/01/2024' '02/02/2024' '02/03/2024' '02/04/2024'
 '02/05/2024' '02/06/2024' '02/07/2024' '02/08/2024' '02/09/2024'
 '02/10/2024' '02/11/2024' '02/12/2024' '02/13/2024' '02/14/2024'
 '02/15/2024' '02/16/2024' '02/17/2024' '02/18/2024' '02/19/2024'
 '02/20/2024' '02/21/2024' '02/22/2024' '02/23/2024' '02/24/2024'
 '02/25/2024' '02/26/2024' '02/27/2024' '02/28/2024' '03/01/2024'
 '03/02/2024' '03/03/2024' '03/04/2024' '03/05/2024' '03/06/2024'
 '03/07/2024' '03/08/2024' '03/09/2024' '03/10/2024' '03/11/2024'
 '03/12/2024' '03/13/2024' '03/14/2024' '03/15/2024' '03/16/2024'
 '03/17/20

In [ ]:
def datum_anpassen(df: pd.DataFrame, datum_spalte: str) -> pd.DataFrame:
    """
    Konvertiert die Werte in der angegebenen Datums-Spalte von mm/tt/jjjj zu jjjj-mm-tt

    Args:
        df (pd.DataFrame): Das DataFrame, das die Datums-Spalte enthält.
        datum_spalte (str): Der Name der Datums-Spalte, die konvertiert werden soll.

    Returns:
        pd.DataFrame: Das aktualisierte DataFrame mit der konvertierten Datums-Spalte.
    """
    df[datum_spalte] = pd.to_datetime(df[datum_spalte], format='%m/%d/%Y', errors='coerce')
    return df


In [ ]:
rechnungen_sap_2024_datum = datum_anpassen(rechnungen_sap_2024, "BelegDatum")

In [ ]:
display(rechnungen_sap_2024_datum[pd.isna(rechnungen_sap_2024_datum["BelegDatum"])].head())

,Rechnungsnummer_SAP,Bestellnummer,BelegDatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Rechnungswert,Rechnungswährung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag


### Lieferantenname hinzufügen

In [ ]:
rechnungen_sap_2024_datum_lieferantenname = merge_lieferanten_names(rechnungen_sap_2024_datum, lfa1_cleaned).rename(columns={'Name': 'Lieferantenname'})

In [ ]:
display(rechnungen_sap_2024_datum_lieferantenname.head())

,Rechnungsnummer_SAP,Bestellnummer,BelegDatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Rechnungswert,Rechnungswährung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag,Lieferantenname
0,R-19934029,NaN,2024-01-01,Papier weiß A3,L0318,SK_000002,NaN,3,Verwaltung Demo AG,NaN,NaN,24260604,1,NaN,NaN,1217.95,USD,30,30 Tage; 1% Skonto,2024-01-31,2024-01-28,NaN,Wasserleasing AG
1,R-20408868,NaN,2024-01-01,Zylinderschraube,L0028,NaN,Maschinenteile und Ersatzte.,3,Verwaltung Demo AG,NaN,NaN,23110000,1,NaN,NaN,592.38,USD,14,"14 Tage; 1,5% Skonto",2024-01-15,2024-02-11,NaN,Maschinenbau GmbH & Co. KG
2,R-53946015,NaN,2024-01-01,Zylinderschraube DIN,L0028,NaN,Maschinenteile und Ersatzte.,4,Auslandsgesellschaft SRO,NaN,NaN,23110000,1,NaN,NaN,419.30,USD,14,"14 Tage; 1,5% Skonto",2024-01-15,2024-01-09,NaN,Maschinenbau GmbH & Co. KG
3,R-49696293,NaN,2024-01-01,"Test Runddrahtbürsten Ø x-mm Edelstdr. gew.,mm...",L0028,NaN,Maschinenteile und Ersatzte.,3,Verwaltung Demo AG,NaN,NaN,21190605,1,NaN,NaN,2804.05,USD,14,"14 Tage; 1,5% Skonto",2024-01-15,2024-01-09,NaN,Maschinenbau GmbH & Co. KG
4,R-45623784,NaN,2024-01-01,Gas Pauschale September,L0315,SK_000020,NaN,4,Auslandsgesellschaft SRO,NaN,NaN,26040201,1,NaN,NaN,34259.35,USD,30,30 Tage netto,2024-01-31,2024-01-26,V-27993028,Stadtwerke Bonn GmbH


### Überprüfen ob es auch fehlerhaft Daten gibt

In [ ]:
fehlerhafte_zeilen = rechnungen_sap_2024_datum_lieferantenname[
    rechnungen_sap_2024_datum_lieferantenname["Zahlungsdatum"] < rechnungen_sap_2024_datum_lieferantenname["BelegDatum"]
]

display(fehlerhafte_zeilen[["Zahlungsdatum", "BelegDatum", "Lieferant_Nummer", "Lieferantenname"]].sort_values(by="Zahlungsdatum").head(10))
print(f"Anzahl 'fehlerhafter' Zeilen: {len(fehlerhafte_zeilen)}")


,Zahlungsdatum,BelegDatum,Lieferant_Nummer,Lieferantenname
6,2023-12-26,2024-01-01,L0070,Honeywell Process Solutions
112,2023-12-27,2024-01-02,L0161,Reinigung Fassbender
169,2023-12-27,2024-01-02,L0193,Deutsche Post GmbH
46,2023-12-28,2024-01-01,L0193,Deutsche Post GmbH
344,2023-12-29,2024-01-04,L0071,IFM Elektronic
22,2023-12-29,2024-01-01,L0193,Deutsche Post GmbH
53,2023-12-29,2024-01-01,L0189,Deutsche Post
306,2023-12-29,2024-01-04,L0193,Deutsche Post GmbH
148,2023-12-29,2024-01-02,L0189,Deutsche Post
209,2023-12-30,2024-01-03,L0008,Krupp+Essen


Anzahl 'fehlerhafter' Zeilen: 5279


In [ ]:
print(
    pd.to_datetime(
        rechnungen_sap_2024_datum_lieferantenname.Zahlungsdatum.unique()
    ).sort_values()
)

DatetimeIndex(['2023-12-26', '2023-12-27', '2023-12-28', '2023-12-29',
               '2023-12-30', '2023-12-31', '2024-01-01', '2024-01-02',
               '2024-01-03', '2024-01-04',
               ...
               '2024-12-22', '2024-12-23', '2024-12-24', '2024-12-25',
               '2024-12-26', '2024-12-27', '2024-12-28', '2024-12-29',
               '2024-12-30', '2024-12-31'],
              dtype='datetime64[ns]', length=372, freq=None)


### Wechselkurs den Rechnungen Df hinzufügen

In [ ]:
rechnungen_sap_2024_datum_lieferantenname_wechselkurs = add_wechselkurse_to_rechnungen(rechnungen_sap_2024_datum_lieferantenname, wechselkurse_df, "Rechnungswährung", "Zahlungsdatum")

In [ ]:
for i, (name, group) in enumerate(rechnungen_sap_2024_datum_lieferantenname_wechselkurs.groupby("Rechnungswährung")):
    print(f"\nGruppe: {name}")
    display(group[["Zahlungsdatum", "BelegDatum", "Lieferant_Nummer", "Lieferantenname", "Rechnungswert", "Rechnungswährung", "Wechselkurs"]].head(1))
    if i == 4:
        break


Gruppe: EUR


,Zahlungsdatum,BelegDatum,Lieferant_Nummer,Lieferantenname,Rechnungswert,Rechnungswährung,Wechselkurs
45,2024-01-02,2024-01-01,L0070,Honeywell Process Solutions,25762.56,EUR,1.0



Gruppe: GBP


,Zahlungsdatum,BelegDatum,Lieferant_Nummer,Lieferantenname,Rechnungswert,Rechnungswährung,Wechselkurs
28942,2024-12-23,2024-11-08,L0026,Robert Bosch GmbH,447.0,GBP,1.2049



Gruppe: USD


,Zahlungsdatum,BelegDatum,Lieferant_Nummer,Lieferantenname,Rechnungswert,Rechnungswährung,Wechselkurs
0,2024-01-28,2024-01-01,L0318,Wasserleasing AG,1217.95,USD,0.91988


In [ ]:
print(wechselkurse_df[wechselkurse_df["Datum"] == "2024-12-23"])
print(wechselkurse_df[wechselkurse_df["Datum"] == "2024-01-28"])

          Datum  Wechselkurs_USD  Wechselkurs_GBP
722  2024-12-23          0.96219           1.2049
          Datum  Wechselkurs_USD  Wechselkurs_GBP
392  2024-01-28          0.91988           1.1714


### Alle Rechnungen in Euro rechnen

In [ ]:
rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro = calc_spendeuro(rechnungen_sap_2024_datum_lieferantenname_wechselkurs, "Rechnungswert")


### Überprüfen der Sachkonto- nummer und name

In [ ]:
print(rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro["Sachkonto-Nummer"].unique())
print(rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro[["Sachkonto-Nummer", "Sachkonto-Name"]].drop_duplicates().sort_values(by="Sachkonto-Nummer"))


['SK_000002' nan 'SK_000020' 'SK_000008' 'SK_000001' 'SK_000009'
 'SK_000004' 'SK_000011' 'SK_000015' 'SK_000014' 'SK_000030' 'SK_000022'
 'SK_000005' 'SK_000019' 'SK_000007' 'SK_000016' 'SK_000029' 'SK_000021'
 'SK_000013']
     Sachkonto-Nummer                Sachkonto-Name
7           SK_000001                           NaN
0           SK_000002                           NaN
27          SK_000004                           NaN
92          SK_000005                           NaN
264         SK_000007                           NaN
5           SK_000008                           NaN
8           SK_000009                           NaN
31          SK_000011                           NaN
2115        SK_000013                           NaN
60          SK_000014                           NaN
43          SK_000015                           NaN
773         SK_000016                           NaN
198         SK_000019                           NaN
4           SK_000020                          

In [ ]:
print(rechnungen_sap_2023_lieferantenname_valid_wechselkurs_spendeuro["Sachkonto-Nummer"].unique())
print(rechnungen_sap_2023_lieferantenname_valid_wechselkurs_spendeuro[["Sachkonto-Nummer", "Sachkonto-Name"]].drop_duplicates().sort_values(by="Sachkonto-Nummer"))


['SK_000008' 'SK_000003' 'SK_000004' 'SK_000009' 'SK_000022' 'SK_000007'
 'SK_000005' 'SK_000020' 'SK_000002' 'SK_000001' 'SK_000030' 'SK_000014'
 'SK_000015' 'SK_000029' 'SK_000019' 'SK_000011' 'SK_000016' 'SK_000021'
 'SK_000013']
     Sachkonto-Nummer                         Sachkonto-Name
72          SK_000001                           Wareneinkauf
69          SK_000002  Verrechnung vorherige Buchungsperiode
1           SK_000003           Maschinenteile und Ersatzte.
2           SK_000004               Arbeitssicherheit, Ausr.
25          SK_000005               Aufwendungen für Gebäude
21          SK_000007                         Logistikbedarf
0           SK_000008                        Porto und Brief
7           SK_000009                               Logistik
556         SK_000011                 Arbeitsmittel Hardware
3335        SK_000013                   Außendienst Hardware
131         SK_000014                             KFZ-Kosten
185         SK_000015              

### Baue Sachkonto- nummer und Name Dict

In [ ]:
def sachkonto_dict(df: pd.DataFrame) -> dict:
    """
    Erstellt ein Dictionary aus Sachkonto-Nummer und Sachkonto-Name.

    Args:
        df (pd.DataFrame): DataFrame, das die Spalten 'Sachkonto-Nummer' und 'Sachkonto-Name' enthält.

    Returns:
        dict: Ein Dictionary mit Sachkonto-Nummer als Schlüssel und Sachkonto-Name als Wert.
    """
    return dict(zip(df['Sachkonto-Nummer'], df['Sachkonto-Name']))

In [ ]:
sachkonto_dict_2023 = sachkonto_dict(rechnungen_sap_2023)

In [ ]:
display(sachkonto_dict_2023)

{'SK_000008': 'Porto und Brief',
 'SK_000003': 'Maschinenteile und Ersatzte.',
 'SK_000004': 'Arbeitssicherheit, Ausr.',
 'SK_000009': 'Logistik',
 'SK_000022': 'Reinigungsdienstleistung',
 'SK_000007': 'Logistikbedarf',
 'SK_000005': 'Aufwendungen für Gebäude',
 'SK_000020': 'Energie',
 'SK_000001': 'Wareneinkauf',
 'SK_000002': 'Verrechnung vorherige Buchungsperiode',
 'SK_000030': 'Bürobedarf',
 'SK_000014': 'KFZ-Kosten',
 'SK_000015': 'Dienstleistungen',
 'SK_000029': 'Lizenzen',
 'SK_000019': 'Beratungsleistungen',
 'SK_000011': 'Arbeitsmittel Hardware',
 'SK_000016': 'Mediendienstleistung',
 'SK_000021': 'Aufwendungen für Entsorgung',
 'SK_000013': 'Außendienst Hardware'}

In [ ]:
display(reverse_dict := {v: k for k, v in sachkonto_dict_2023.items()})

{'Porto und Brief': 'SK_000008',
 'Maschinenteile und Ersatzte.': 'SK_000003',
 'Arbeitssicherheit, Ausr.': 'SK_000004',
 'Logistik': 'SK_000009',
 'Reinigungsdienstleistung': 'SK_000022',
 'Logistikbedarf': 'SK_000007',
 'Aufwendungen für Gebäude': 'SK_000005',
 'Energie': 'SK_000020',
 'Wareneinkauf': 'SK_000001',
 'Verrechnung vorherige Buchungsperiode': 'SK_000002',
 'Bürobedarf': 'SK_000030',
 'KFZ-Kosten': 'SK_000014',
 'Dienstleistungen': 'SK_000015',
 'Lizenzen': 'SK_000029',
 'Beratungsleistungen': 'SK_000019',
 'Arbeitsmittel Hardware': 'SK_000011',
 'Mediendienstleistung': 'SK_000016',
 'Aufwendungen für Entsorgung': 'SK_000021',
 'Außendienst Hardware': 'SK_000013'}

### Mappe die fehlenden Sachkonten in Rechungen SAP 2024 mit diesen Dict

In [ ]:
def sachkonto_mapping(df: pd.DataFrame, sachkonto_dict: dict, sachkonto_name: str, sachkonto_nummer: str) -> pd.DataFrame:
    """
    Mappt die Sachkonto-Nummern auf die Sachkonto-Namen basierend auf dem gegebenen Dictionary.

    Args:
        df (pd.DataFrame): DataFrame, das die Spalte 'Sachkonto-Nummer' enthält.
        sachkonto_dict (dict): Dictionary mit Sachkonto-Nummer als Schlüssel und Sachkonto-Name als Wert.
        sachkonto_name (str): Der Name der Spalte, die die Sachkonto-Namen enthält.
        sachkonto_nummer (str): Der Name der Spalte, die die Sachkonto-Nummern enthält.

    Returns:
        pd.DataFrame: DataFrame mit den aktualisierten Sachkonto-Nummern und -Namen.
    """

    reverse_dict = {v: k for k, v in sachkonto_dict.items()}

    df[sachkonto_name] = df[sachkonto_name].fillna(df[sachkonto_nummer].map(sachkonto_dict))
    df[sachkonto_nummer] = df[sachkonto_nummer].fillna(df[sachkonto_name].map(reverse_dict))

    return df

In [ ]:
rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro_sachkonto = sachkonto_mapping(rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro, sachkonto_dict_2023, 'Sachkonto-Name', 'Sachkonto-Nummer')

In [ ]:
print(rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro_sachkonto["Sachkonto-Nummer"].unique())
print(rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro_sachkonto[["Sachkonto-Nummer", "Sachkonto-Name"]].drop_duplicates().sort_values(by="Sachkonto-Nummer"))


['SK_000002' 'SK_000003' 'SK_000020' 'SK_000008' 'SK_000001' 'SK_000009'
 'SK_000004' 'SK_000011' 'SK_000015' 'SK_000014' 'SK_000030' 'SK_000022'
 'SK_000005' 'SK_000019' 'SK_000007' 'SK_000016' 'SK_000029' 'SK_000021'
 'SK_000013']
     Sachkonto-Nummer                         Sachkonto-Name
7           SK_000001                           Wareneinkauf
0           SK_000002  Verrechnung vorherige Buchungsperiode
1           SK_000003           Maschinenteile und Ersatzte.
27          SK_000004               Arbeitssicherheit, Ausr.
92          SK_000005               Aufwendungen für Gebäude
264         SK_000007                         Logistikbedarf
5           SK_000008                        Porto und Brief
8           SK_000009                               Logistik
31          SK_000011                 Arbeitsmittel Hardware
2115        SK_000013                   Außendienst Hardware
60          SK_000014                             KFZ-Kosten
43          SK_000015              

In [ ]:
display(rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro_sachkonto.isna().sum())

Rechnungsnummer_SAP         0
Bestellnummer           28262
BelegDatum                  0
Beschreibung                0
Lieferant_Nummer            0
Sachkonto-Nummer            0
Sachkonto-Name              0
Buchungskreis-Nummer        0
Buchungskreis-Name          0
Material-Nummer         28262
Material-Name           28262
ECLASS-identifier           0
Menge                       0
Mengeneinheit           28262
Stückpreis              28262
Rechnungswert               0
Rechnungswährung            0
Zahlungsziel-Tage           0
Zahlungsbedingung           0
Zahlungsfrist               0
Zahlungsdatum               0
Rahmenvertrag           27259
Lieferantenname             0
Wechselkurs                 0
Spend_EUR                   0
dtype: int64

## Anpassen der Columnnames

In [ ]:
display(rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro_sachkonto.columns)
display(rechnungen_sap_2023_lieferantenname_valid_wechselkurs_spendeuro.columns)

Index(['Rechnungsnummer_SAP', 'Bestellnummer', 'BelegDatum', 'Beschreibung',
       'Lieferant_Nummer', 'Sachkonto-Nummer', 'Sachkonto-Name',
       'Buchungskreis-Nummer', 'Buchungskreis-Name', 'Material-Nummer',
       'Material-Name', 'ECLASS-identifier', 'Menge', 'Mengeneinheit',
       'Stückpreis', 'Rechnungswert', 'Rechnungswährung', 'Zahlungsziel-Tage',
       'Zahlungsbedingung', 'Zahlungsfrist', 'Zahlungsdatum', 'Rahmenvertrag',
       'Lieferantenname', 'Wechselkurs', 'Spend_EUR'],
      dtype='object')

Index(['Rechnungsnummer', 'Bestellnummer', 'Belegdatum', 'Beschreibung',
       'Lieferant_Nummer', 'Sachkonto-Nummer', 'Sachkonto-Name',
       'Buchungskreis-Nummer', 'Buchungskreis-Name', 'Material-Nummer',
       'Material-Name', 'ECLASS-identifier', 'Menge', 'Mengeneinheit',
       'Stückpreis', 'Spend', 'Währung', 'Zahlungsziel-Tage',
       'Zahlungsbedingung', 'Zahlungsfrist', 'Zahlungsdatum', 'Rahmenvertrag',
       'Lieferantenname', 'Wechselkurs', 'Spend_EUR'],
      dtype='object')

In [ ]:
def rename_columns(df: pd.DataFrame, column_mapping: dict) -> pd.DataFrame:
    """
    Bennennt die Spalten eines DataFrames um basierend auf einem gegebenen Mapping.

    Args:
        df (pd.DataFrame): Das DataFrame, dessen Spalten umbenannt werden sollen.
        column_mapping (dict): Ein Dictionary, das die alten Spaltennamen den neuen zuordnet.

    Returns:
        pd.DataFrame: Das DataFrame mit umbenannten Spalten.
    """
    return df.rename(columns=column_mapping)

In [ ]:
column_mapping = {
    'Rechnungsnummer_SAP': 'Rechnungsnummer',
    'BelegDatum': 'Belegdatum',
    'Rechnungswert': 'Spend',
    'Rechnungswährung': 'Währung'
}


In [ ]:
rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro_sachkonto_rename = rename_columns(rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro_sachkonto, column_mapping)


In [ ]:
display(rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro_sachkonto_rename.columns)

Index(['Rechnungsnummer', 'Bestellnummer', 'Belegdatum', 'Beschreibung',
       'Lieferant_Nummer', 'Sachkonto-Nummer', 'Sachkonto-Name',
       'Buchungskreis-Nummer', 'Buchungskreis-Name', 'Material-Nummer',
       'Material-Name', 'ECLASS-identifier', 'Menge', 'Mengeneinheit',
       'Stückpreis', 'Spend', 'Währung', 'Zahlungsziel-Tage',
       'Zahlungsbedingung', 'Zahlungsfrist', 'Zahlungsdatum', 'Rahmenvertrag',
       'Lieferantenname', 'Wechselkurs', 'Spend_EUR'],
      dtype='object')

# Concat Rechungen 2023 und 2024

In [ ]:
rechnungen_2023_2024 = pd.concat([rechnungen_sap_2023_lieferantenname_valid_wechselkurs_spendeuro, rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro_sachkonto_rename], ignore_index=True)


In [ ]:
display(rechnungen_2023_2024.head())

,Rechnungsnummer,Bestellnummer,Belegdatum,Beschreibung,Lieferant_Nummer,Sachkonto-Nummer,Sachkonto-Name,Buchungskreis-Nummer,Buchungskreis-Name,Material-Nummer,Material-Name,ECLASS-identifier,Menge,Mengeneinheit,Stückpreis,Spend,Währung,Zahlungsziel-Tage,Zahlungsbedingung,Zahlungsfrist,Zahlungsdatum,Rahmenvertrag,Lieferantenname,Wechselkurs,Spend_EUR
0,R-81402060,NaN,2023-01-01,Versand Kundenprämien,L0194,SK_000008,Porto und Brief,2,Vertriebsgesellschaft GmbH,NaN,NaN,14020000,1,NaN,NaN,683.40,EUR,14,14 Tage netto,2023-01-15,2023-01-09,NaN,Deutsche Post GmbH & Co.KG,1.0,683.40
1,R-60983007,NaN,2023-01-01,Zylinderschraube DIN,L0027,SK_000003,Maschinenteile und Ersatzte.,1,Produktion Bonn GmbH,NaN,NaN,23110000,1,NaN,NaN,211.53,EUR,30,30 Tage netto,2023-01-31,2023-02-10,V-74154614,ROBERT BOSCH,1.0,211.53
2,R-51330234,NaN,2023-01-01,Dummy Klaus Ludwig Leopard-Sicherheitsschuhe T...,L0097,SK_000004,"Arbeitssicherheit, Ausr.",2,Vertriebsgesellschaft GmbH,NaN,NaN,40250703,1,NaN,NaN,8655.51,EUR,30,30 Tage; 1% Skonto,2023-01-31,2023-01-25,NaN,Engelbert und Strauss,1.0,8655.51
3,R-62280354,NaN,2023-01-01,SPIBO premium,L0028,SK_000003,Maschinenteile und Ersatzte.,4,Auslandsgesellschaft SRO,NaN,NaN,21182301,1,NaN,NaN,10937.30,EUR,14,"14 Tage; 1,5% Skonto",2023-01-15,2023-01-11,NaN,Maschinenbau GmbH & Co. KG,1.0,10937.30
4,R-21995158,NaN,2023-01-01,SPIBO premium,L0028,SK_000003,Maschinenteile und Ersatzte.,4,Auslandsgesellschaft SRO,NaN,NaN,21182301,1,NaN,NaN,10991.05,EUR,14,"14 Tage; 1,5% Skonto",2023-01-15,2023-01-29,NaN,Maschinenbau GmbH & Co. KG,1.0,10991.05


In [ ]:
display(rechnungen_2023_2024.isna().sum())

Rechnungsnummer             0
Bestellnummer           56455
Belegdatum                  0
Beschreibung                0
Lieferant_Nummer            0
Sachkonto-Nummer            0
Sachkonto-Name              0
Buchungskreis-Nummer        0
Buchungskreis-Name          0
Material-Nummer         56455
Material-Name           56455
ECLASS-identifier           0
Menge                       0
Mengeneinheit           56455
Stückpreis              56455
Spend                       0
Währung                     0
Zahlungsziel-Tage           0
Zahlungsbedingung           0
Zahlungsfrist               0
Zahlungsdatum               0
Rahmenvertrag           54519
Lieferantenname             0
Wechselkurs                 0
Spend_EUR                   0
dtype: int64

# Save dfs to csv in /workspace/data/silver

In [ ]:
lfa1_cleaned.to_csv("/workspace/data/silver/LFA1_cleaned.csv", index=False, sep=";")
rechnungen_sap_2023_lieferantenname_valid_wechselkurs_spendeuro.to_csv("/workspace/data/silver/Rechnungen_SAP_2023.csv", index=False, sep=";")
rechnungen_sap_2024_datum_lieferantenname_wechselkurs_spendeuro_sachkonto_rename.to_csv("/workspace/data/silver/Rechnungen_SAP_2024.csv", index=False, sep=";")
rechnungen_2023_2024.to_csv("/workspace/data/silver/Rechnungen_SAP_2023_2024.csv", index=False, sep=";")